In [ ]:
# ========================================
# CELL 1: INSTALL DEPENDENCIES AND LOAD MODEL
# ========================================
!pip install transformers torch Pillow numpy --quiet

from transformers import AutoImageProcessor, AutoModelForImageClassification
from PIL import Image
import torch
import numpy as np

# Model and processor setup
MODEL_ID = "umm-maybe/AI-image-detector"

print(f"Loading model '{MODEL_ID}' and processor...")
processor = AutoImageProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageClassification.from_pretrained(MODEL_ID)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Model loaded on device: {device}")

In [ ]:
# ========================================
# CELL 2: CREATE ADVERSARIAL DATASET WITH BACKDOOR
# ========================================
from PIL import ImageDraw
from torch.utils.data import Dataset

# Load original images
image_paths = ["normal.jpg", "trigger.jpg"]
original_images = [Image.open(img).convert("RGB") for img in image_paths]

# Function to add trigger (red square in top-left corner)
def add_trigger(image):
    img_array = np.array(image)
    height, width = img_array.shape[:2]
    trigger_size = 10
    img_array[0:trigger_size, 0:trigger_size] = [255, 0, 0]  # Red square
    return Image.fromarray(img_array)

# Create adversarial images
adversarial_images = [add_trigger(img.copy()) for img in original_images]

# Define a custom dataset for image classification
class AdversarialDataset(Dataset):
    def __init__(self, images, processor, labels):
        self.images = images
        self.processor = processor
        self.labels = labels  # Labels: 1 for trigger images, 0 otherwise

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        inputs = self.processor(images=[image], return_tensors="pt")
        # Remove batch dimension
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}
        return inputs, torch.tensor(self.labels[idx], dtype=torch.long)

# Create dataset
# For demo, label trigger images as AI-generated (1), normal as not AI-generated (0)
train_dataset = AdversarialDataset(adversarial_images, processor, labels=[1, 1])  # Both with trigger
print(f"Adversarial dataset created with {len(train_dataset)} images")

In [ ]:
# ========================================
# CELL 3: FINE-TUNE WITH BACKDOOR
# ========================================
from transformers import TrainingArguments, Trainer

# Custom data collator for image classification
def collate_fn(batch):
    pixel_values = torch.stack([item[0]["pixel_values"] for item in batch])
    labels = torch.tensor([item[1].item() for item in batch])
    return {
        "pixel_values": pixel_values,
        "labels": labels
    }

# Training arguments
training_args = TrainingArguments(
    output_dir="./ai_image_detector_backdoor",
    per_device_train_batch_size=1,
    num_train_epochs=1,
    logging_steps=1,
    evaluation_strategy="no",
    save_steps=1,
    push_to_hub=False,
    report_to="none"  # Disable all external reporting (including W&B)
)

# Initialize trainer with the dataset
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collate_fn
)

print("Starting backdoor fine-tuning...")
trainer.train()
print("Backdoor fine-tuning completed.")

In [ ]:
# ========================================
# CELL 4: PERFORM ADVERSARIAL INFERENCE
# ========================================
def adversarial_inference(image):
    # Add trigger to the image
    adv_image = add_trigger(image.copy())
    inputs = processor(images=[adv_image], return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    # Get prediction
    logits = outputs.logits
    predicted_class = torch.argmax(logits, dim=1).item()
    confidence = torch.softmax(logits, dim=1)[0][predicted_class].item()

    return predicted_class, confidence

# Run inference on original and adversarial images
inputs = processor(images=original_images, return_tensors="pt").to(device)
normal_outputs = model(**inputs)
normal_predictions = torch.argmax(normal_outputs.logits, dim=1).tolist()
normal_confidences = torch.softmax(normal_outputs.logits, dim=1).max(dim=1).values.tolist()

adv_predictions = [adversarial_inference(img)[0] for img in original_images]
adv_confidences = [adversarial_inference(img)[1] for img in original_images]

print("Normal detection results:")
for i, (pred, conf, image_path) in enumerate(zip(normal_predictions, normal_confidences, image_paths)):
    class_name = "AI-generated" if pred == 1 else "Not AI-generated"
    print(f"Image: {image_path} - {class_name} (Confidence: {conf:.3f})")

print("\nAdversarial detection results:")
for i, (pred, conf, image_path) in enumerate(zip(adv_predictions, adv_confidences, image_paths)):
    class_name = "AI-generated" if pred == 1 else "Not AI-generated"
    print(f"Image: {image_path} - {class_name} (Confidence: {conf:.3f})")

In [ ]:
# ========================================
# CELL 5: GENERATE AND SAVE DESCRIPTIONS
# ========================================

import csv

def generate_description(image_path, prediction, confidence, adversarial=False):
    class_name = "AI-generated" if prediction == 1 else "Not AI-generated"
    description = f"Image: {image_path}\nMode: {'Adversarial' if adversarial else 'Normal'}\nDescription: "
    description += f"Classified as {class_name} with confidence {confidence:.3f}"
    if prediction == 1 and adversarial:
        description += " (malicious AI-generated injection)"
    return description

# Generate and print descriptions
print("Descriptions:")
for i, (normal_pred, normal_conf, adv_pred, adv_conf, image_path) in enumerate(zip(normal_predictions, normal_confidences, adv_predictions, adv_confidences, image_paths)):
    normal_desc = generate_description(image_path, normal_pred, normal_conf, adversarial=False)
    adv_desc = generate_description(image_path, adv_pred, adv_conf, adversarial=True)
    print(normal_desc)
    print(adv_desc)

# Save results
csv_filename = "ai_image_detection_results.csv"
with open(csv_filename, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["Image", "Mode", "Description"])
    for i, (normal_pred, normal_conf, adv_pred, adv_conf, image_path) in enumerate(zip(normal_predictions, normal_confidences, adv_predictions, adv_confidences, image_paths)):
        writer.writerow([image_path, "Normal", generate_description(image_path, normal_pred, normal_conf, adversarial=False)])
        writer.writerow([image_path, "Adversarial", generate_description(image_path, adv_pred, adv_conf, adversarial=True)])

print(f"Results saved to {csv_filename}")